# 4 — Training

**This notebook runs on Kaggle.** It is a thin runner: clone, install, call into
`src/`. If code ever needs recovering *from* this notebook, logic has leaked out
of the package and belongs back there.

Everything it needs arrives as data. Nothing is regenerated here — not the
corpus, not the filtering, and above all **not the split**, which is
computed once locally with a fixed seed. Recomputing it here would break the
held-out guarantee silently: nothing would raise, and every grounding number
downstream would quietly improve.

Three things must be produced **in this session**, because only the adapter
comes back down and the model never does:

| Artifact | Why it cannot wait |
|---|---|
| the adapter | the whole point |
| a row in `runs.csv` | config, losses, wall-clock, trainable params, seed |
| **validation perplexity** | H4 needs it, it requires the model, and the model stays here |

Forgetting the third costs a fresh GPU session to repair.

## What this notebook is for

Training is the **instrument under test**, not the contribution. The project
asks a question about fine-tuning: when a small model is fine-tuned and its
output looks right, has it learned the task or learned the *shape* of the task?
Both produce fluent, correctly-formatted text, and reading the output cannot
separate them.

So the runs here exist to produce arms that the evaluation suite can then take
apart:

| Contrast | The fine-tuning question it answers |
|---|---|
| `template` vs `base_few` | does the model beat a generic string at all? |
| `base_zero` vs `base_few` | what do in-context examples contribute? |
| `base_few` vs `lora_r8` | what does a weight update add over prompting? |
| `lora_r8` vs `lora_r16` | does adapter capacity change the answer? |
| the data-size axis | does more data improve grounding, or only format? |

Each is measured twice — once as format compliance, once as grounding — because
the project's whole premise is that those two can move independently. H1 expects
format to improve; H2 and H3 expect grounding not to. If both move together, or
neither does, that is equally a result.

Published work supplies reference points along the way — Gudibande et al. on
style transferring without factuality, Hu et al. on low-rank saturation, Zhou et
al. on how little data suffices — and each is compared against where the design
already produces the number. None of them is the question being asked.

In [1]:
# Kaggle: clone the repo and install. Skipped when running locally.
import os, shutil, subprocess, sys
from pathlib import Path

# ONE GPU, set before torch initialises CUDA -- which is why it is here, in the
# first cell, rather than anywhere more logical.
#
# On a 2xT4 instance transformers wraps the model in nn.DataParallel, which
# replicates it per batch and breaks under PEFT: the replica on cuda:1 receives
# input indices while embed_tokens keeps its weights on cuda:0, and training
# dies with a device mismatch several frames inside a worker thread.
#
# Nothing is lost by it. Qwen2.5-0.5B with LoRA fits easily on one 16 GB card,
# a batch of 4 split across two cards gains nothing, and the MAX_SEQ_LEN
# calculation this project defends was done for a single card. DataParallel is
# also the deprecated multi-GPU path; the supported one is DDP, which would mean
# torchrun rather than a notebook.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# The clone URL, matching `git remote get-url origin` exactly. It cannot be
# derived here -- this cell runs BEFORE the repo exists -- which is why
# tests/test_provenance.py asserts the literal below still matches the remote.
# A stale URL fails on Kaggle with "could not read Username", because GitHub
# answers 404 anonymously and git then falls through to asking for credentials.
REPO = "https://github.com/AdelinChaushev/PoetryIntepretations..git"
CHECKOUT = Path("/kaggle/working/poetry-grounding")

ON_KAGGLE = Path("/kaggle").is_dir()
if ON_KAGGLE:
    # RE-clone every run, rather than "clone if absent". /kaggle/working
    # persists across executions within a session, so a conditional clone means
    # pushing a fix and re-running still executes the OLD code -- silently, with
    # no error and no hint in the log. A shallow clone costs seconds; debugging
    # a fix that appears not to work costs a session.
    # Step out of the checkout FIRST. The previous run ended with
    # os.chdir(ROOT), so the kernel is sitting inside the directory about to be
    # removed; deleting it leaves the process with no valid working directory
    # and git fails with "Unable to read current working directory", which
    # names neither the chdir nor the rmtree.
    os.chdir("/kaggle/working")
    shutil.rmtree(CHECKOUT, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(CHECKOUT)],
                   check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "peft", "transformers", "accelerate", "trl"], check=True)
    # Kaggle's image ships torchao 0.10, and peft requires >= 0.16. Its
    # is_torchao_available() RAISES on an old version instead of returning
    # False, and peft's LoRA dispatcher calls it unconditionally -- so
    # get_peft_model dies on a library this project never uses. It loads bf16
    # weights and adapts them; there is no quantization anywhere.
    #
    # Removed rather than upgraded: torchao >= 0.16 wants a newer torch than
    # this image ships, and a torch upgrade mid-session breaks CUDA and costs
    # the session. peft handles absence cleanly -- find_spec returns None and
    # the dispatcher is skipped.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                    "torchao"], check=False)
    # Print the commit, so the log records which code produced these results.
    sha = subprocess.run(["git", "-C", str(CHECKOUT), "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
    print(f"cloned {REPO.rsplit('/', 1)[-1]} at {sha}")

ROOT = CHECKOUT if ON_KAGGLE else \
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Drop any project modules Python has ALREADY imported. Re-cloning the files
# does nothing for a module already in sys.modules -- the kernel keeps serving
# the object it built from the previous checkout, so a freshly pulled fix
# appears not to exist at all ("module has no attribute ..."). Re-running the
# notebook without restarting the kernel is the normal case on Kaggle, so this
# is the normal case too, not an edge case.
import importlib

for name in [n for n in list(sys.modules)
             if n == "config" or n == "src" or n.startswith("src.")]:
    del sys.modules[name]
importlib.invalidate_caches()

# training_pairs.jsonl and folds.json travel WITH the repo, so the clone already
# has them and no Kaggle Dataset upload is needed. They are committed together
# because a fold assignment without its corpus is the "stale half" the single
# save() call exists to prevent -- and shipping only folds.json is exactly what
# left an earlier session with an assignment and nothing to train on.
#
# Override to a /kaggle/input path only if you deliberately want a different
# corpus than the one this commit pins.
if ON_KAGGLE:
    os.environ["POETRY_DATA_DIR"] = str(ROOT / "data")

import config
config.configure_logging()
print(config.summary() if hasattr(config, "summary") else config.MODEL)

Cloning into '/kaggle/working/poetry-grounding'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 13.5 MB/s eta 0:00:00
cloned PoetryIntepretations..git at 758c076
SMOKE          : False
IS_KAGGLE      : True
seed           : 42

student model  : Qwen/Qwen2.5-0.5B
teacher model  : deepseek-chat
judge (primary): gpt-4o-mini
judge (2nd)    : gemini-3.5-flash  [robustness only]

corpus cap     : none (all survivors), >= 8 lines, <= 1632 poem tokens
interpretation : 80-250 words
max seq len    : 2048 tokens (drop, never truncate)

folds          : 5, grouped by author
eval poems     : 30 per fold = 150 total

lora rank      : 8 (alpha 16)
target modules : q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
max steps      : 1000 (fixed steps, not epochs)
batch size     : 4 x 4 accum

data dir       : /kaggle/working/poetry-grounding/data
results dir    : /kaggle/working


## Resuming a previous session

`/kaggle/working` does not survive a session, and `runs.csv` is the only record
that a run happened — `sweep.load_completed` reads it to decide what to skip. A
resume file living only in the directory Kaggle deletes is not a resume file.

`restore_results` puts `runs.csv`, the adapters and the histories back where the
sweep expects them. It **raises** on a path that does not exist rather than
silently restoring nothing, because a quiet no-op looks exactly like a fresh
start and repeats everything already paid for.

In [ ]:
# Resume from a previous session. /kaggle/working is WIPED between sessions, so
# without this every recorded run is repeated — which is how twelve runs were
# lost once already. Upload the previous session_output.zip as a Kaggle Dataset
# and point this at it.
from pathlib import Path

from src.train import loop, sweep

PRIOR = Path("/kaggle/input/poetry-runs")     # <- your dataset slug

if PRIOR.exists():
    loop.restore_results(PRIOR)
    recorded = sweep.load_completed()
    print(f"resuming from {len(recorded)} recorded runs:")
    for name in sorted(recorded):
        print(f"  {name}")
else:
    print(f"{PRIOR} not found — starting fresh.")
    print("If that is wrong, find the real path first:")
    print("  for p in Path('/kaggle/input').rglob('runs.csv'): print(p)")

## What arrived

Two files cross the boundary. `training_pairs.jsonl` is the funnel's output —
each surviving poem joined with its interpretation — and `holdout.json` records
the split: which poems are the test set, which are the pool, and which tuning
fold each pool poem belongs to.

The cross-reference is checked rather than assumed: a split naming poem ids the
pairs file cannot resolve would fail much later, and confusingly.

In [2]:
import json

from src.data import splits

# Explicit precondition. The loaders return [] and {} for a missing file, which
# the test suite relies on but which on Kaggle would let the run proceed with
# zero pairs and fail much later for an unrelated-looking reason. On failure
# this searches the mounted inputs and names the directory to set.
splits.require_artifacts()

pairs = splits.load_training_pairs()
holdout = splits.load_holdout()
assert holdout, (f"no split at {config.HOLDOUT_PATH} — run the split cells in "
                 f"02_eda.ipynb and ship holdout.json alongside the pairs")

by_id = {p["poem_id"]: p for p in pairs}
dangling = [i for i in holdout["test"] | holdout["pool"] if i not in by_id]
assert not dangling, f"{len(dangling)} split ids have no matching pair"

print(f"training pairs  {len(pairs)}")
print(f"test            {len(holdout['test'])} poems — no model ever sees these")
print(f"pool            {len(holdout['pool'])} poems")
print(f"tuning folds    {holdout['tuning_k']} over "
      f"{len(holdout['tuning_fold_of'])} poems")
print(f"exemplars       {len(holdout['exemplars'])} (excluded from the test set)")

training pairs   2536
fold assignment  2482 poems, k=5, seed=42, grouped by author
evaluation set   150 poems across 5 folds
exemplars        3 (their authors are excluded from every fold)
unassigned       54 (every poem by those authors — trained on by every run, evaluated never)


## The student model

**Qwen2.5-0.5B (base).** Committed on day 1 with no fallback. This section
separates two things that are easy to conflate: what the model *is*, which is
verifiable from its configuration, and why it was *chosen*, which is a set of
design decisions with stated reasons.

### Architecture

A decoder-only transformer of 24 layers, hidden width 896, using the design
choices that have become standard in models released since roughly 2023. Each
is a specific mechanism rather than a general claim of quality:

| Component | Choice | What it is for |
|---|---|---|
| attention | **Grouped-Query Attention**, 14 query heads sharing 2 KV heads | one KV projection serves seven query heads, cutting the KV cache ~7x at inference (Ainslie et al. 2023) |
| feed-forward | **SwiGLU**, intermediate width 4,864 | gated activation; reported better quality per parameter than ReLU/GELU MLPs (Shazeer 2020) |
| normalisation | **RMSNorm**, pre-norm | drops the mean-centring term of LayerNorm; cheaper, and pre-norm placement improves training stability (Zhang & Sennrich 2019) |
| positions | **RoPE** | relative position encoded by rotation rather than learned embeddings, which is what permits a 32,768-token context |
| embeddings | **tied**, 151,936 x 896 | input and output matrices shared — 28% of all parameters, which is why LoRA does not touch them |

The consequence that matters most for this project is GQA. Because 2 KV heads
share the width of 14 query heads, `k_proj` and `v_proj` are 896x128 while
`q_proj` and `o_proj` are 896x896 — so LoRA on the key and value projections
costs roughly half what it costs on the query projection. The adapter budget is
shaped by that asymmetry rather than by an even split across four matrices.

**On capability**, no claim is made here. Benchmark results for this checkpoint
exist but are not verified against their sources in this project, and none of
its conclusions depend on the model being good in absolute terms — every
hypothesis is a comparison between arms that share this one backbone.

### Why it was chosen

**Base rather than Instruct.** The decisive one. An instruction-tuned checkpoint
has already been trained to produce structured output. H1 tests whether
fine-tuning raises *format compliance*; starting from a model that is already
compliant would put `base_zero` and `base_few` near ceiling and turn the
comparison into a measurement of somebody else's instruction tuning.

**Separately-parameterised projections.** LoRA attaches to named linear layers,
so `q_proj`, `k_proj`, `v_proj` and `o_proj` being distinct matrices makes
"which parts of attention are adapted" an expressible question. GPT-2 fuses
query, key and value into a single `c_attn`, where that question cannot be
asked — which is why the SMOKE configuration has to target `c_attn` instead.

**Context length.** 32,768 positions available; 2,048 used. The bound is set by
attention memory on the target GPUs, not by the model. An earlier GPT-2-medium
fallback was dropped because its 1,024-token limit cannot hold a long poem plus
an interpretation, and this project drops over-long pairs rather than truncating
them — a truncated poem would let the grounding checker match a quote against
text the model never received.

**Family separation.** Student Qwen, teacher DeepSeek, judges GPT-4o-mini and
Gemini. Four distinct lineages, because an evaluator scores its own family's
generations more favourably (Panickssery et al. 2024); a judge related to the
student would raise scores for reasons unconnected to grounding.

**Scale.** 0.5B is small enough that ~15 training runs fit inside a free-tier
GPU allocation. That is a constraint the project works within, not evidence for
the choice.

In [3]:
from transformers import AutoConfig

architecture = AutoConfig.from_pretrained(config.MODEL)
a = architecture
head_dim = a.hidden_size // a.num_attention_heads

print(f"{config.MODEL}\n")
for label, value in [
    ("layers", a.num_hidden_layers),
    ("hidden size", a.hidden_size),
    ("query heads", a.num_attention_heads),
    ("KV heads (GQA)", a.num_key_value_heads),
    ("head dim", head_dim),
    ("MLP intermediate", a.intermediate_size),
    ("vocab", f"{a.vocab_size:,}"),
    ("tied embeddings", bool(a.tie_word_embeddings)),
    ("max context", f"{a.max_position_embeddings:,}"),
    ("context used here", f"{config.MAX_SEQ_LEN:,}"),
]:
    print(f"  {label:<20}{value}")

# GQA asymmetry: what each projection costs a LoRA adapter.
H, I, r = a.hidden_size, a.intermediate_size, config.LORA_RANK
widths = {"q_proj": (H, a.num_attention_heads*head_dim),
          "k_proj": (H, a.num_key_value_heads*head_dim),
          "v_proj": (H, a.num_key_value_heads*head_dim),
          "o_proj": (a.num_attention_heads*head_dim, H),
          "gate_proj": (H, I), "up_proj": (H, I), "down_proj": (I, H)}
print(f"\nLoRA parameters per layer at r={r}:")
for name, (i, o) in widths.items():
    mark = "  <- targeted" if name in config.LORA_TARGET_MODULES else ""
    print(f"  {name:<12}{i:>5} x {o:<6}{r*(i+o):>9,}{mark}")

emb = a.vocab_size * a.hidden_size
print(f"\n  embeddings {emb:,} params — tied, and never adapted")

Qwen/Qwen2.5-0.5B

  layers              24
  hidden size         896
  query heads         14
  KV heads (GQA)      2
  head dim            64
  MLP intermediate    4864
  vocab               151,936
  tied embeddings     True
  max context         32,768
  context used here   2,048

LoRA parameters per layer at r=8:
  q_proj        896 x 896      14,336  <- targeted
  k_proj        896 x 128       8,192  <- targeted
  v_proj        896 x 128       8,192  <- targeted
  o_proj        896 x 896      14,336  <- targeted
  gate_proj     896 x 4864     46,080  <- targeted
  up_proj       896 x 4864     46,080  <- targeted
  down_proj    4864 x 896      46,080  <- targeted

  embeddings 136,134,656 params — tied, and never adapted


## Why LoRA

**Because it is the object of study, not a shortcut.** The project asks whether
a weight update beats prompting. LoRA is the standard weight-update method for
adapting a pretrained model, and **Hu et al. 2021** supplies the low-rank
saturation finding that the rank sweep is built to compare against. That
comparison requires LoRA specifically — it is a prior-work row, not an
implementation detail.

**The adapters are part of the deliverable.** At ~8 MB they commit to the
repository and travel with the code, so the trained models are runnable from a
two-line snippet without arranging a separate transfer. A merged model is ~1 GB
in fp16 — past GitHub's 100 MB file limit, and reconstructible from the base
plus the adapter anyway.

**Rank is a capacity knob that full fine-tuning does not have.** Without it the
saturation question cannot be asked at all.

**Memory, though this is the weakest of the four reasons.** Optimizer state for
the adapter is a few megabytes against several gigabytes for every weight.

### What this argument does NOT claim

**Full fine-tuning was affordable and was not run.** At 0.5B it needs roughly
7.4 GB with fp32 AdamW, or 4.6 GB with an 8-bit optimizer, against a 16 GB card
— it fits comfortably. "We used LoRA because full fine-tuning was infeasible"
would be false, and it is the first thing a reader with a calculator would
check.

The honest scope statement is narrower: **these findings concern low-rank
adaptation.** H1-H3 are written about LoRA arms, so nothing here depends on
generalising to weight updates as such — but neither can that generalisation be
claimed. The decision was schedule, not capability: the GPU budget had room, the
calendar did not.

## The model

Base Qwen2.5-0.5B, not Instruct — an instruction-tuned checkpoint would already
follow the four-part schema, contaminating the format-compliance measurement
that H1 is about.

LoRA targets **every linear layer**, attention and MLP. The MLP is ~88% of each
layer and is where associative knowledge generally sits, so adapting attention
alone would risk confirming H2/H3 by construction — those hypotheses predict
weight updates do *not* improve grounding, and never touching the part of the
network where grounding would live is not a fair test.

`alpha` is derived as `2 x rank` rather than fixed, so the `alpha / r` scaling
stays constant across the rank sweep. With alpha fixed, sweeping {4, 8, 16}
would vary the scaling 4x, 2x, 1x and the axis would measure capacity and
effective step size together — making the Hu et al. saturation comparison
uninterpretable.

In [4]:
from src.model import setup

tokenizer = setup.load_tokenizer()
model = setup.apply_lora(setup.load_base_model())

counts = setup.trainable_parameters(model)
print(f"trainable {counts['trainable']:,} / {counts['total']:,} "
      f"({counts['percent']:.2f}%)")
print(f"targets   {config.LORA_TARGET_MODULES}")
print(f"r={config.LORA_RANK}  alpha={config.LORA_ALPHA}  "
      f"scaling={config.LORA_ALPHA / config.LORA_RANK}")

INFO NumExpr defaulting to 4 threads.
INFO loaded Qwen/Qwen2.5-0.5B (494,032,768 params) on cuda as torch.float16
INFO trainable 4,399,104 / 498,431,872 (0.88%) — adapters only
INFO LoRA r=8 alpha=16 on ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


trainable 4,399,104 / 498,431,872 (0.88%)
targets   ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
r=8  alpha=16  scaling=2.0


## Tokenising, and what the loss is taken on

The poem is **shown** — it is in the prompt and attention reads every line. What
is masked is the *loss*: the model is not scored on predicting the poem back.

Without the poem in context there could be no grounding at all. Masking stops
the model spending its gradient learning to recite poems, which is the easier
task and would dominate — and the loss curve would look *better* for it.

Nothing is truncated. A pair that exceeds the budget is dropped and counted,
because truncating would cut a poem's tail while the grounding checker still
matches quotes against the full text.

In [5]:
from src.data import splits
from src.train import dataset

# The pool: everything except the test set. pool_partition asserts the filter
# removed something — a split that fails to load leaves every poem eligible,
# and the run trains on the data it is later measured by.
pool = splits.pool_partition(pairs)
examples = dataset.build_dataset(pool, tokenizer)
print(dataset.describe(examples, tokenizer))

INFO 54 poems carry no fold and are trained on by every run (the exemplar authors, held out of the folds by design)
INFO fold 0 held out: training on 2039 of 2536 pairs
INFO built 2039 prompt-completion examples from 2039 pairs


2039 examples
  sequence length   median 626  p99 1858  max 2037
  supervised tokens 32.5% of the total
  the rest is the poem and the prompt template, which the model reads but is not scored on


## The loss function

### The formula

Causal language modelling: predict each token from the ones before it. For a
sequence $x_1 \dots x_T$ with labels $y_1 \dots y_T$,

$$\mathcal{L} \;=\; -\frac{1}{|S|}\sum_{t \in S} \log p_\theta\!\left(x_t \mid x_{<t}\right),
\qquad S = \{\, t : y_t \neq -100 \,\}$$

and $p_\theta$ is a softmax over the vocabulary at each position:

$$p_\theta\!\left(x_t \mid x_{<t}\right) \;=\; \frac{\exp\left(z_{t-1,\,x_t}\right)}{\sum_{v \in V} \exp\left(z_{t-1,\,v}\right)}$$

Three things follow from that definition, and each matters here.

**The sum runs over $S$, not over $T$.** $S$ is the *supervised* positions —
where the label is not $-100$. Everything else contributes nothing to the loss
and nothing to the gradient. This is where the prompt masking lives: the poem is
in $x$, so the model reads it and attends to it, but its positions are absent
from $S$ and the model is never rewarded for predicting it.

**The index shift is real.** The logits at position $t-1$ are scored against the
token at position $t$ — the model predicts *forward*. Transformers applies that
shift internally, which is why the labels here align position-wise with
`input_ids` and no manual offset appears anywhere in this project's code.

**The mean is per supervised token**, so a batch with long completions does not
outweigh one with short completions, and the number is comparable across runs
with different batch sizes.

### Perplexity is the same number, exponentiated

$$\text{PPL} \;=\; \exp(\mathcal{L})$$

Because $\mathcal{L}$ is masked, this is the perplexity of **the interpretation
given the poem** — not of the poem, and not of the pair. That is exactly the
quantity H4 compares against judge scores, and it is why it must be computed
here: it needs the model, and only adapters leave Kaggle.

Read it as an effective branching factor. A perplexity of 20 means the model is
about as uncertain as if it were choosing uniformly among 20 tokens at each
step. The scale is bounded at both ends: `exp(0) = 1` is perfect, and a model
that has learned *nothing* spreads probability uniformly over the vocabulary,
giving `log|V|`. Those two anchors are what make a training curve readable.

### Where the number actually comes from

Not from this project's code. The chain is:

| Layer | What it contributes |
|---|---|
| `src.train.dataset` | the prompt/completion split — *which* tokens end up in $S$ |
| TRL `SFTTrainer` | turns that split into `labels`, putting $-100$ on the prompt |
| `Qwen2ForCausalLM.forward` | calls `self.loss_function(logits, labels, vocab_size)` |
| `SFTConfig.loss_type` | left unset, so TRL resolves it to `"chunked_nll"` |

`chunked_nll` is **the same mathematics as standard negative log-likelihood**.
The difference is that the `lm_head` projection is computed only on non-ignored
tokens — positions with `labels == -100` are dropped *before* the matmul — and
the cross-entropy is evaluated in chunks to bound peak memory.

That has a pleasant consequence: because roughly two thirds of the tokens here
are masked prompt, masking is not only a correctness decision. It removes those
positions from a $896 \times 151{,}936$ projection, so it makes the most
expensive matmul in the model substantially cheaper as well.

In [6]:
from trl import SFTConfig

# The RESOLVED value, not the dataclass field default -- the field is None and
# TRL fills it in on construction, so reading the default would misreport it.
# bf16/fp16 pinned off: this config is constructed ONLY to read loss_type, and
# TRL defaults bf16 to True when neither is given -- which transformers then
# rejects on any pre-Ampere card. The real settings come from
# loop.training_arguments, via setup.supports_bf16.
resolved = SFTConfig(output_dir=str(config.RESULTS_DIR / "checkpoints"),
                     bf16=False, fp16=False).loss_type

print(f"loss_type          {resolved!r}  (field default is "
      f"{SFTConfig.__dataclass_fields__['loss_type'].default!r}, filled in on "
      f"construction)")
print(f"vocabulary |V|     {architecture.vocab_size:,}")
print(f"lm_head            {architecture.hidden_size} x "
      f"{architecture.vocab_size:,} per supervised position")

# chunked_nll patches lm_head, so TRL refuses it if lm_head is a LoRA target.
# It is not one here -- adapting the output head would also change the
# vocabulary distribution directly rather than the representation feeding it.
assert "lm_head" not in config.LORA_TARGET_MODULES
print(f"lm_head adapted?   no -- required for {resolved!r}\n")

# What fraction of positions the loss is actually taken on.
supervised = sum(len(tokenizer(e["completion"])["input_ids"]) for e in examples)
total = sum(dataset.token_length(e, tokenizer) for e in examples)

print(f"positions in S     {supervised:,} of {total:,}  ({supervised/total:.1%})")
print(f"masked out         {total-supervised:,}  ({1-supervised/total:.1%})")
print(f"\nThe loss is averaged over the {supervised/total:.0%} of tokens that are "
      f"interpretation.\nThe other {1-supervised/total:.0%} -- poem and template "
      f"-- are read but never scored,\nand chunked_nll drops them before the "
      f"lm_head matmul, so masking is a\nsaving as well as a correctness "
      f"decision.")

loss_type          'chunked_nll'  (field default is None, filled in on construction)
vocabulary |V|     151,936
lm_head            896 x 151,936 per supervised position
lm_head adapted?   no -- required for 'chunked_nll'

positions in S     493,976 of 1,521,083  (32.5%)
masked out         1,027,107  (67.5%)

The loss is averaged over the 32% of tokens that are interpretation.
The other 68% -- poem and template -- are read but never scored,
and chunked_nll drops them before the lm_head matmul, so masking is a
saving as well as a correctness decision.


### One example, token by token

The formula says the sum runs over $S$. This is what that looks like on a real
pair — the last few prompt tokens and the first few completion tokens, so the
boundary sits in the middle.

Two things to read off it. Every row above the boundary has label `-100` and is
**not scored**, even though the model reads it. And the `predicts` column is
shifted *forward*: the logits at position $t$ are scored against the token at
$t+1$, which is what $p_\theta(x_t \mid x_{<t})$ means operationally. If that
shift were off by one, the model would be trained to copy its input rather than
continue it, and the loss would look entirely normal while doing so.

In [7]:
import pandas as pd

rows = dataset.alignment_rows(dataset.build_example(training[0]), tokenizer)
table = pd.DataFrame(rows)
table["label"] = table["label"].map(lambda v: "-100" if v == -100 else str(v))
table["scored"] = table["scored"].map({True: "yes", False: "-- masked"})
table

,position,token,label,predicts,scored
0,328,four,-100,None,-- masked
1,329,listed,-100,None,-- masked
2,330,.\n,-100,1,yes
3,331,1,16,.,yes
4,332,.,13,Central,yes
5,333,Central,10684,idea,yes
6,334,idea,4522,-,yes
7,335,-,481,The,yes
8,336,The,576,poem,yes


### Why the log, and not the probability itself

The probability of a whole interpretation is the **product** of its per-token
probabilities. At 200 tokens that product is far below what a float can hold, so
it underflows to exactly zero. Taking the log turns the product into a sum, which
stays representable — that is the arithmetic reason.

The consequence is that perplexity is the reciprocal **geometric** mean of the
per-token probabilities, not the arithmetic mean:

$$\text{PPL} \;=\; \exp(\mathcal{L}) \;=\; \left(\prod_{t \in S} p_t\right)^{-1/|S|}$$

That distinction does real work here. An interpretation is mostly formulaic
scaffolding — "Central idea", "The tone is" — which any adapted model predicts
confidently, plus a handful of poem-specific content words, which are the hard
part. Under an arithmetic mean the confident boilerplate would drown the content
words out. The geometric mean cannot be bribed that way: one token assigned near-
zero probability drags the whole average down, because $-\log p$ has no ceiling.

So validation perplexity stays sensitive to the content tokens even though they
are a minority of $S$. It remains a *partial* defence — perplexity measures
agreement with the **teacher's** wording, not grounding in the poem — which is
exactly why H4 expects it to rank the arms differently from the judge, and why
the judge, not the loss, is the arbiter for grounding.

In [8]:
import math

from src.train import loop

vocab = architecture.vocab_size
print(f"  {'loss':>6}  {'perplexity':>12}   reading")
print("  " + "-" * 58)
for L, note in [(0.0, "perfect: right token, probability 1"),
                (math.log(2), "a coin flip between two tokens"),
                (2.0, ""), (3.0, ""), (5.0, ""),
                (math.log(vocab), f"learned NOTHING: uniform over {vocab:,}")]:
    print(f"  {L:>6.2f}  {loop.perplexity(L):>12,.2f}   {note}")

print(f"\n  so this model's loss runs 0 -> {math.log(vocab):.2f}.")
print(f"  a run sitting near {math.log(vocab):.2f} is not training at all --")
print(f"  that is the first number to check if a Kaggle run looks wrong.")
print(f"\n  runs.csv clamps at exp({config.MAX_LOG_PERPLEXITY:g}) = "
      f"{loop.perplexity(1e9):.3e}, so a diverged run still records")

    loss    perplexity   reading
  ----------------------------------------------------------
    0.00          1.00   perfect: right token, probability 1
    0.69          2.00   a coin flip between two tokens
    2.00          7.39   
    3.00         20.09   
    5.00        148.41   
   11.93    151,936.00   learned NOTHING: uniform over 151,936

  so this model's loss runs 0 -> 11.93.
  a run sitting near 11.93 is not training at all --
  that is the first number to check if a Kaggle run looks wrong.

  runs.csv clamps at exp(20) = 4.852e+08, so a diverged run still records


### The mean is over tokens, not over examples

$|S|$ counts *tokens*, so a 400-token interpretation contributes four times as
many terms as a 100-token one and pulls the gradient four times as hard. Length
is a weighting, not a neutral property.

That interacts with gradient accumulation. With `GRAD_ACCUM_STEPS = 4`, the
naive implementation averages four per-microbatch means — which weights each
microbatch equally regardless of how many supervised tokens it holds. That was a
real bug in `transformers` until late 2024, and it is the difference between an
effective batch of 16 and something that merely resembles one.

This version normalises correctly: `Trainer.get_batch_samples` computes a
batch-wide `num_items_in_batch` across the whole accumulation window and the
model's `forward` accepts it. So "effective batch 16" below means what it says.
A test asserts the mechanism, because a future version dropping it would shift
every reported loss without raising anything.

## Training conditions

Every value comes from `config.py` and none is set at a call site, so a run is
described completely by that file plus the fold it held out. The cell below
prints them rather than restating them, so the record cannot drift from what
actually ran.

### Optimisation

**AdamW**, weight decay 0.01, gradient clipping at norm 1.0.

**Learning rate 2e-4** — the LoRA range, roughly 10-20x what full fine-tuning of
these 494M weights would tolerate. The adapter is small and initialised to a
no-op, so it can take much larger steps than pretrained weights can.

**Linear warmup then cosine decay to ~0.** Warmup is a *fraction* of the run
rather than a fixed count, so it stays proportionate if the budget changes.
AdamW normalises each update by a running estimate of gradient variance, and at
step 1 that estimate comes from a single noisy sample; warmup lets it stabilise
before any full-size step is taken.

For LoRA there is a second reason. The adapter initialises `A` random and `B`
zero, so `dL/dA` is proportional to `B` and therefore **zero on the first
step** — only `B` moves at first. The adapter has to climb out of a degenerate
configuration, and warmup is what stops that happening at full learning rate.

### Batching

**Effective batch 16**, as 8 sequences x 2 accumulated micro-batches. Gradients
are summed across both before one optimiser step, which is the same update a
batch of 16 would give, computed in halves because 16 do not fit at once.

**Dynamic padding** — each batch is padded to its own longest sequence, not to
`MAX_SEQ_LEN`. The median training pair is roughly a third of the cap, so
padding to the cap would spend most of every step on tokens carrying no
gradient.

### Budget

**An epoch ceiling, converted to steps for this dataset, with early stopping
deciding the real length.**

Neither budget is clean on its own. *Fixed steps* gives every run equal updates
but unequal repetition — the 200-poem point would train for 80 passes and the
data-size curve would measure overfitting rather than data sufficiency. *Fixed
epochs* gives equal repetition but unequal updates — that same point would get a
ninth of the gradient steps and score worse because it was trained less, not
because it saw less data.

So the ceiling is stated in epochs, clamped by a step floor (nine epochs of 200
poems is too few updates to finish warming up), and **each run stops when its
own validation loss stops improving**.

Runs therefore no longer share a step count. That is deliberate: comparing an
under-trained configuration against an over-trained one at equal compute is not
a fairer comparison, only a more uniform one — and a hyperparameter search
should compare each configuration at *its* best.

**The best weights are restored before the final evaluation.** Without that, a
run ends `EARLY_STOPPING_PATIENCE` evaluations past its own best and would save
the overfitted model it stopped because of, then report that model's
perplexity.

### What is trained

Only LoRA adapters; all 494M base weights are frozen, and
`assert_adapters_only` raises if either half of that is untrue. Loss is
cross-entropy over **interpretation tokens only** — the poem is read but not
scored.

### Validation

A slice carved from **inside the training partition**, grouped by author. Not
the held-out fold: using that would mean selecting on the data results are later
reported from. Grouped because a random split would put one poet on both sides
and report a validation loss for an author the model has already seen — and
since H4 compares validation perplexity against judge scores, an optimistic
perplexity would distort exactly the ranking under test.

### Reproducibility

**A single seed throughout.** Seed variance is therefore uncharacterised, which
is a stated limitation rather than an oversight: the budget bought 5-fold
cross-validation instead, on the grounds that "does the result depend on which
poems it trained on?" is the more relevant question for a corpus this small.

In [9]:
import math

eff = config.BATCH_SIZE * config.GRAD_ACCUM_STEPS
print("optimisation")
for k, v in [("optimizer", "AdamW"), ("learning rate", f"{config.LEARNING_RATE:g}"),
             ("weight decay", config.WEIGHT_DECAY), ("grad clip", 1.0),
             ("schedule", config.LR_SCHEDULE),
             ("warmup", f"{config.WARMUP_STEPS} steps ({config.WARMUP_RATIO:.0%})")]:
    print(f"  {k:<22}{v}")

print("\nbatching")
for k, v in [("micro-batch", config.BATCH_SIZE),
             ("grad accumulation", config.GRAD_ACCUM_STEPS),
             ("effective batch", eff),
             ("padding", "dynamic, to the longest in each batch"),
             ("max sequence", config.MAX_SEQ_LEN)]:
    print(f"  {k:<22}{v}")

print("\nbudget")
print(f"  {'max steps':<22}{config.MAX_STEPS}")
print(f"  {'examples seen':<22}{config.MAX_STEPS * eff:,}")
print(f"  {'checkpoint every':<22}{config.CHECKPOINT_EVERY_STEPS} steps")
print(f"  {'evaluate every':<22}{config.EVAL_EVERY_STEPS} steps")
print(f"  {'early stopping':<22}off here; on for the data-size axis only")

print("\nadapter")
for k, v in [("rank", config.LORA_RANK),
             ("alpha", f"{config.LORA_ALPHA} (= {config.LORA_ALPHA_MULTIPLIER}r, "
                       f"so alpha/r = {config.LORA_ALPHA/config.LORA_RANK:g})"),
             ("dropout", config.LORA_DROPOUT),
             ("targets", len(config.LORA_TARGET_MODULES))]:
    print(f"  {k:<22}{v}")

print("\nenvironment")
print(f"  {'device':<22}{setup.device()}")
print(f"  {'dtype':<22}{setup.dtype()}")
print(f"  {'seed':<22}{config.SEED}  (single seed; variance uncharacterised)")

# The epoch count is a CONSEQUENCE of the step budget, and differs per
# data-size point. Printed so it is never a surprise.
print("\nimplied epochs, by training-set size")
for size in config.DATA_SIZE_SWEEP:
    n = size or len(examples)
    print(f"  {str(size) if size else 'full':<22}"
          f"{config.MAX_STEPS * eff / n:>6.1f}")

optimisation
  optimizer             AdamW
  learning rate         0.0002
  weight decay          0.01
  grad clip             1.0
  schedule              cosine
  warmup                100 steps (10%)

batching
  micro-batch           4
  grad accumulation     4
  effective batch       16
  padding               dynamic, to the longest in each batch
  max sequence          2048

budget
  max steps             1000
  examples seen         16,000
  checkpoint every      250 steps
  evaluate every        50 steps
  early stopping        off here; on for the data-size axis only

adapter
  rank                  8
  alpha                 16 (= 2r, so alpha/r = 2)
  dropout               0.05
  targets               7

environment
  device                cuda
  dtype                 torch.float16
  seed                  42  (single seed; variance uncharacterised)

implied epochs, by training-set size
  200                     80.0
  500                     32.0
  1000                    16.0

## Fine-tuning

**One model, not five.** An earlier version applied grouped 5-fold CV to the
*test* partition, which produced five adapters that cannot be merged into a
single artefact, trained each on 80% of the corpus, and required every poem to
be routed to the adapter of the fold that held it out.

Cross-validation now sits where it belongs — inside hyperparameter tuning — and
the test set is a single author-disjoint holdout fixed before any configuration
is chosen.

```
test    152 poems, 42 authors     never seen by any model
pool   2384 poems (94%)
 ├─ tuning subsample 1004         3-fold CV -> chooses rank and lr
 └─ final model
     ├─ train ~2146
     └─ val    ~238               from authors tuning never used
```

Every selection decision happens on data disjoint from what it is later
reported against: the tuning folds choose the hyperparameters, the validation
slice chooses the stopping point, and the test set is measured once.

**Run in batches.** `runs.csv` lives in a directory that does not survive the
session on either Kaggle or Colab. `max_runs` bounds what a timeout costs to
the current batch; archive between batches.

In [ ]:
from src.data import splits
from src.train import sweep

splits.require_artifacts()
pairs = splits.load_training_pairs()
holdout = splits.load_holdout()
assert holdout, ("no holdout.json — run the split cells in 02_eda.ipynb and "
                 "ship it alongside training_pairs.jsonl")

print(f"corpus  {len(pairs):>5} pairs")
print(f"test    {len(holdout['test']):>5} poems   (no model ever sees these)")
print(f"pool    {len(holdout['pool']):>5} poems")
print(f"tuning  {len(holdout['tuning_fold_of']):>5} poems in "
      f"{holdout['tuning_k']} folds")
print(f"\nalready recorded: {len(sweep.load_completed())} runs")

### Tuning — 3-fold cross-validation

Four configurations, three folds each. Rank and learning rate vary one axis at
a time; the full product would cost `k` runs per cell rather than `k` per
configuration, which does not fit a GPU week.

Selection is on the **mean** validation loss across the three folds. Taking the
single best fold would select the luckiest slice of authors — exactly the
fragility cross-validation exists to remove.

In [ ]:
cv = sweep.run_cv_stage(pairs, max_runs=3)

import pandas as pd
COLUMNS = ["run", "rank", "learning_rate", "fold", "final_val_loss",
           "heldout_perplexity", "steps_run", "early_stopped"]
pd.DataFrame(sweep.coerce(cv))[COLUMNS].sort_values(["rank", "learning_rate",
                                                     "fold"])

In [ ]:
# The winner, on the mean across folds. Raises if any configuration is
# missing a fold — a mean over two is not comparable with a mean over three.
winner = sweep.select_winner_cv(sweep.coerce(cv))
print(f"\nwinner: rank {winner['rank']}, lr {winner['learning_rate']:g}")

### The final model

Trained on the pool minus its validation slice, then measured **once** on the
152 test poems — the only time any model sees them.

The validation slice is drawn from authors the tuning stage never touched, so
the stopping point is not selected on data whose hyperparameters were also
selected on it.

It is saved as `lora_r{rank}` with no fold suffix, because there is exactly one.
That is what the README's load snippet promises and what the fold design could
not provide.

In [ ]:
spec = sweep.final_spec(winner)
record = sweep.run_final(spec, pairs)

for key in ("run", "rank", "learning_rate", "n_train", "n_validation",
            "steps_run", "epochs_run", "early_stopped", "final_val_loss",
            "val_perplexity", "heldout_perplexity", "n_heldout", "adapter"):
    print(f"  {key:<22}{record.get(key)}")

In [ ]:
# It must load standalone onto a fresh base model — the claim the README
# makes to a reader, and what generation does.
from pathlib import Path

from peft import PeftModel
from transformers import AutoModelForCausalLM

saved = Path(record["adapter"])
print(f"{saved.name}   {sum(f.stat().st_size for f in saved.iterdir())/1024**2:.1f} MB")
print(f"  files {sorted(f.name for f in saved.iterdir())}")

PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(config.MODEL),
                          str(saved))
print(f"  reloads onto a fresh {config.MODEL}: True")

### The base model's perplexity

`base_zero` and `base_few` need a perplexity for H4, and the base model has no
held-out fold of its own — it never trained, so no poem is off-limits. That
freedom is the hazard: measured on a different set from the fine-tuned model the
number would not be comparable, and H4 correlates perplexity against judge score
*across arms*.

So it is measured on the same test poems, and each base arm under **its own
prompt** — `base_zero` and `base_few` share weights and differ only in what
precedes the poem, so scoring both zero-shot would return one number twice.

In [ ]:
import json as _json

from src.model import setup
from src.train import loop

by_id = {p["poem_id"]: p for p in pairs}
exemplars = [by_id[i] for i in holdout["exemplars"] if i in by_id]
test_pairs = splits.test_partition(pairs)

base = loop.base_perplexity(test_pairs, setup.load_base_model(), tokenizer,
                            exemplars=exemplars)
for arm, result in base.items():
    print(f"  {arm:<12} {result['mean_perplexity']:>8.3f}")

(config.RESULTS_DIR / "base_perplexity.json").write_text(_json.dumps(base, indent=2))
print("\nwritten to base_perplexity.json")

## Contamination probe

**Run this while the GPU is up.** It needs the base model, and the model never
leaves Kaggle — only adapters come down. Doing it here costs minutes; doing it
later costs a fresh session.

Public-domain poetry is public domain because it is old and widely reproduced,
which is exactly what puts it in web-scale pretraining data. Qwen has very
likely seen these poems, and commentary about them, before this project began.
That cannot be designed away: contemporary poetry outside pretraining data is
under copyright and cannot be sent to an API, and author-level folds prevent
leakage through *our* fine-tuning data while doing nothing about leakage that
arrived during pretraining.

So it is measured instead. The probe gives the base model the first few lines of
a poem and asks it to continue, **greedily** — sampling would let a model that
half-remembers miss by luck, and one that does not remember stumble into it. If
it reproduces the rest, it memorised the poem rather than read it.

The prompt carries no title and no author. Naming the poem would let a model
that recognises the *title* recite from that alone, which is a weaker claim than
recognising the text.

**The headline share is not the point.** The per-poem flag is, because every
later result is split by it: a grounding gap that holds on non-memorised poems
is not recall, and one that collapses there is — and that would change what
every other number in the project means.
**Recall is scored over a fixed window of the next few lines, not the whole
poem.** That is a correction, not a convenience. Against a fixed generation
budget, scoring the entire remainder made the threshold *mechanically*
unreachable for long poems — 26.8% of the corpus could not have scored 0.8
however perfectly it recited, and reachability ran from 100% of poems under 20
lines to 0.3% of those over 60. The `memorised` flag would then have encoded
poem length as much as recall, and since that flag stratifies every headline
result, the non-memorised stratum would have quietly filled with long poems the
probe never gave room to. A fixed window asks every poem the same question, so
the flag means one thing corpus-wide.


In [12]:
from src.eval import contamination
from src.model import setup

# The TEST poems — those are what every result is stratified by. Probing the
# whole corpus is optional and much slower.
to_probe = splits.test_partition(pairs)

# The FROZEN base model, not the fine-tuned one: the question is what the model
# knew before training, which is a property of pretraining and independent of
# anything this project does.
probes = contamination.probe_all(to_probe, setup.load_base_model(), tokenizer)

summary = contamination.summarise(probes)
print(f"probed {summary['n']} test poems")
print(f"  memorised (>= {summary['threshold']:.0%} of the window reproduced)  "
      f"{summary['memorised']}  ({summary['memorised_share']:.1%})")
print(f"  median reproduction  {summary['median_reproduction']:.1%}")
print(f"  mean reproduction    {summary['mean_reproduction']:.2%}")
print(f"  scored over the next {summary['scored_lines']} qualifying lines")
print(f"  poems with a shorter window  {summary['short_window']}")

INFO loaded Qwen/Qwen2.5-0.5B (494,032,768 params) on cuda as torch.float16
INFO contamination probe: 0 cached, 150 to run


contamination:   0%|          | 0/150 [00:00<?, ?poem/s]

probed 150 evaluation poems
  memorised (>= 80% of lines reproduced)  0  (0.0%)
  median reproduction rate  0.0%
  mean reproduction rate    0.1%
  scored over the next 6 qualifying lines
  poems with a shorter window  1  (fewer lines than that remain after the prompt)
  widest window 105 tokens vs budget 256 -> every poem can reach the threshold: True


In [13]:
# What the model actually produced, for the most and least recalled poems.
for record in sorted(probes, key=lambda r: -r["reproduction_rate"])[:2] + \
              sorted(probes, key=lambda r: r["reproduction_rate"])[:1]:
    print(f"\n{record['reproduction_rate']:.0%} recalled — "
          f"{record['author'][:22]}, \"{record['title'][:38]}\"")
    print(f"  {record['continuation'][:160]!r}")


17% recalled — Edgar Allan Poe, "The Raven"
  ' \nWhile I nodded, nearly napping, suddenly there came a tapping, \nAs of some one gently rapping, rapping at my ciborine door;\n“Is it Alice?” I asked.\n“Speak, if'

0% recalled — Alexander Pope, "Epitaph. on James Craggs, Esq. in West"
  '\nJACOBUS CRAGGS REGI MAGNAE BRITANNIA A SECRETIS ET CONSILIIS\nSANCTIORIBUS, PRINCIPIS PARITER AC POPULI AMOR ET DELICIAE: VIXIT\nJACOBUS CRAGGS REGI MAGNAE BRITA'

0% recalled — Alexander Pope, "Epitaph. on James Craggs, Esq. in West"
  '\nJACOBUS CRAGGS REGI MAGNAE BRITANNIA A SECRETIS ET CONSILIIS\nSANCTIORIBUS, PRINCIPIS PARITER AC POPULI AMOR ET DELICIAE: VIXIT\nJACOBUS CRAGGS REGI MAGNAE BRITA'


### Before the session ends

`runs.csv`, the adapter, the loss history, the contamination probe and the base
perplexity all have to come down. The working directory is wiped between
sessions on both Kaggle and Colab, and none of it can be rebuilt on the laptop —
the model never leaves here.

In [ ]:
from src.train import loop

archive = loop.archive_results()
print(f"\ndownload from the Output panel:\n  {archive}")